In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# Read csv
def read_csv(path):
    return spark.read.option("header", True).option("inferSchema", True).csv(path)

In [0]:
#Write File
def writefile(df,path,mode):
    df.write.mode(mode).format("csv").options(header='true', inferSchema='true').save(path)

In [0]:
#Ingest Timestamp
def ingest_ts(df):
    return df.withColumn("ingest_ts",from_utc_timestamp(current_timestamp(),"America/Sao_Paulo"))

In [0]:
#update_ts
def update_ts(df):
    return df.withColumn("update_ts",from_utc_timestamp(current_timestamp(),"America/Sao_Paulo"))

In [0]:
# snake_case
def clean_column_names(df):
    return df.toDF(
        *[i.lower().replace(" ", "_").lower() for i in df.columns]
    )

In [0]:
# string space remove
def trim_space_col(df,col_names):
    return df.select([trim(col(i)).alias(i) if i in col_names else col(i)
                           for i in df.columns])



In [0]:
#Drop Duplicates
def drop_duplicates(df,col_name):
    return df.dropDuplicates([col_name])

In [0]:
# string clean
def clean_string_column(df,trim_cols:list=[],lower_cols:list=[],upper_cols:list=[]):
    for i in trim_cols:
        df = df.withColumn(i, trim(col(i)))
    for i in lower_cols:
        df = df.withColumn(i, lower(col(i)))
    for i in upper_cols:
        df = df.withColumn(i, upper(col(i)))
    return df

In [0]:
# cast data

def cast_col(df, col_dict):
    for column_name, data_type in col_dict.items():
        df = df.withColumn(
            column_name,
            expr(f"try_cast(`{column_name}` AS {data_type})")
        )
    return df


In [0]:
# handle null
def handle_null(df,null_dict:dict):
    return df.fillna(null_dict)

In [0]:
#not null
def remove_null(df, not_null_cols:list):
    for i in not_null_cols:
        df= df.filter(col(i).isNotNull())
        return df


In [0]:
def sremove(df,col_names:list,pattern: str=r"[^a-zA-Z0-9\s_-]"):
    for i in col_names:
        df=df.withColumn(i,trim(regexp_replace(regexp_replace(col(i),pattern," "),r"\s+", " "))
        )
        
        return df



In [0]:

def rename_columns(df, rename_dict: dict):
    for i, j in rename_dict.items():
        if i in df.columns:
            df = df.withColumnRenamed(i, j)
    return df